# 05 · Held-out evaluation (ScienceQA test / AI2D)

This scores fixed, already-trained models on a held-out split. Nothing is trained or tuned here. The models and settings were
chosen on the validation split only (`results/reports/scaling/`). Decoding and prompt are identical to every validation run:
greedy, max 160 new tokens, batch 8.

Adapters are downloaded at the start from the training notebooks' outputs (`ADAPTER_FILES`). Those links are short-lived
signed URLs filled in at submission time; they are never committed to git.

Settings: Accelerator → GPU T4, Internet → On.

In [ ]:
# Parameters
DATASET = 'scienceqa'     # "scienceqa" or "ai2d"
SPLIT = 'test'         # "test"
MODELS = [
 {
  "name": "base",
  "config": "base"
 },
 {
  "name": "qlora_n6218",
  "config": "qlora",
  "files": {
   "adapter_config.json": "<signed-url>",
   "adapter_model.safetensors": "<signed-url>"
  }
 },
 {
  "name": "lora_n6218",
  "config": "lora",
  "files": {
   "adapter_config.json": "<signed-url>",
   "adapter_model.safetensors": "<signed-url>"
  }
 }
]

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import json, os, subprocess, sys, time, urllib.request
from pathlib import Path

REPO_DIR = "/tmp/VisionTune"
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "main",
                    "https://github.com/akhilnathreddy/VisionTune.git", REPO_DIR], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, os.path.join(REPO_DIR, "src"))
COMMIT = subprocess.run(["git", "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout.strip()
print(COMMIT)

# Fetch adapters first, so expired links fail in the first minute rather than after setup.
for m in MODELS:
    if not m.get("files"):
        continue
    d = Path("/tmp/adapters") / m["name"]
    d.mkdir(parents=True, exist_ok=True)
    for fname, url in m["files"].items():
        urllib.request.urlretrieve(url, d / fname)
    m["adapter"] = str(d)
    print(m["name"], {p.name: p.stat().st_size for p in d.iterdir()})

In [ ]:
!pip install -q -U "transformers>=4.57" "peft>=0.17" "accelerate>=1.0" "datasets>=3.0" "bitsandbytes>=0.45" qwen-vl-utils
!pip install -q -e .

In [ ]:
OUT = Path("/kaggle/working") / f"heldout_{DATASET}_{SPLIT}"
(OUT / "predictions").mkdir(parents=True, exist_ok=True)
timings = {}

def sh(cmd):
    print("$", cmd, flush=True)
    t0 = time.perf_counter()
    subprocess.run(cmd, shell=True, check=True)
    return round(time.perf_counter() - t0, 1)

skip = "--skip-ai2d" if DATASET == "scienceqa" else ""
timings["prepare_s"] = sh(f"python -m visiontune.data.prepare --config configs/base.yaml {skip} > /dev/null")
print(json.loads(Path("data/processed/prepare_report.json").read_text()).get(DATASET if DATASET == "ai2d" else "scienceqa", {}).get("splits", {}).get(SPLIT, ""))

In [ ]:
results = {}
for m in MODELS:
    pred = OUT / "predictions" / f"{m['name']}_{DATASET}_{SPLIT}.jsonl"
    adapter = f"--adapter {m['adapter']}" if m.get("adapter") else ""
    timings[f"{m['name']}_s"] = sh(f"python -m visiontune.inference.predict --config configs/{m['config']}.yaml {adapter} "
                                   f"--dataset {DATASET} --split {SPLIT} --out {pred}")
    met = json.loads(pred.with_suffix(".metrics.json").read_text())
    results[m["name"]] = {k: met.get(k) for k in ("overall", "format", "by_subject", "by_near_dup_in_train",
                                                  "latency_s_per_example_batched", "mean_new_tokens",
                                                  "generated_tokens_per_s_batched", "peak_vram_gb")}
    print(m["name"], met["overall"], met["format"], flush=True)
(OUT / "result.json").write_text(json.dumps({"dataset": DATASET, "split": SPLIT, "commit": COMMIT,
                                             "timings_s": timings, "models": results}, indent=2))